# Racing Gym RL — Results Demo

Loads pre-committed artifacts from the repo. No environment install, no training required.

**Runtime**: CPU is fine. Runs in under 2 minutes.

| Section | What it shows |
|---|---|
| 1. Setup | Clone repo, confirm tracked files |
| 2. PPO results | Benchmark table + AutoResearch chart |
| 3. PPO videos | Agent control behavior |
| 4. World model | Reward-faithfulness benchmark table |
| 5. RSSM arch | Parameter count and model structure |
| 6. World model videos | Hallucination comparison |

In [ ]:
!git clone https://github.com/yuvimalik/Racing_Gym_RL.git 2>/dev/null || true
%cd Racing_Gym_RL
!git pull origin main --quiet

from pathlib import Path
import json, sys

artifacts = [
    "autoresearch/results/best_metrics.json",
    "autoresearch/results/experiments.jsonl",
    "demo_assets/world_model_benchmark_summary.json",
    "docs/assets/video1.mp4",
    "docs/assets/video2.mp4",
    "docs/assets/video3.mp4",
    "docs/assets/video4.mp4",
]
for p in artifacts:
    status = "ok" if Path(p).exists() else "MISSING"
    print(f"  {status:8}  {p}")

## 2. PPO Results

In [ ]:
import pandas as pd

best = json.loads(Path("autoresearch/results/best_metrics.json").read_text())
recursive = json.loads(Path("autoresearch/results/recursive_cap100/promoted/metrics.json").read_text())

pd.DataFrame([
    {"run": "run_008 (best AutoResearch)",
     "reward": round(best["mean_reward"], 2),
     "progress": f"{best['mean_progress']:.1%}",
     "off_track": f"{best['offtrack_rate']:.1%}",
     "speed": round(best["mean_speed"], 1),
     "timesteps": best["total_timesteps"]},
    {"run": "g003_c03 (best recursive)",
     "reward": round(recursive["mean_reward"], 2),
     "progress": f"{recursive['mean_progress']:.1%}",
     "off_track": f"{recursive['offtrack_rate']:.1%}",
     "speed": round(recursive["mean_speed"], 1),
     "timesteps": recursive["total_timesteps"]},
])

In [ ]:
import matplotlib.pyplot as plt

experiments = [json.loads(l) for l in open("autoresearch/results/experiments.jsonl") if l.strip()]
valid = [e for e in experiments if e.get("mean_reward", -9999) > -900]
ids     = [e["experiment_id"] for e in valid]
rewards = [e["mean_reward"] for e in valid]
prog    = [e.get("mean_progress", 0) * 100 for e in valid]
is_best = [e.get("was_best", False) for e in valid]
colors  = ["#2ecc71" if b else "#3498db" for b in is_best]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.bar(ids, rewards, color=colors)
ax1.axhline(0, color="gray", lw=0.8, ls="--")
ax1.set_xlabel("Experiment"); ax1.set_ylabel("Mean Reward")
ax1.set_title("AutoResearch reward per run (green = new best)")
ax2.bar(ids, prog, color=colors)
ax2.set_xlabel("Experiment"); ax2.set_ylabel("Track Progress (%)")
ax2.set_title("Track progress per run")
plt.suptitle(f"{len(valid)} runs  |  {sum(is_best)} promoted to new best", y=1.02)
plt.tight_layout(); plt.show()

## 3. PPO Agent Videos

**Video 1** — single-car PPO after the control stack stabilised.
**Video 2** — multi-car shared-policy evaluation.

In [ ]:
from IPython.display import Video, display

for label, path in [
    ("Video 1 — PPO single-car control", "docs/assets/video1.mp4"),
    ("Video 2 — Multi-car shared-policy", "docs/assets/video2.mp4"),
]:
    print(label)
    display(Video(path, embed=True, html_attributes="controls loop width=640"))
    print()

## 4. World Model Benchmark

Reward and telemetry faithfulness across three checkpoints.
All numbers from .

In [ ]:
summary = json.loads(Path("demo_assets/world_model_benchmark_summary.json").read_text())
print(summary["qualitative_summary"]["main_takeaway"])
print()
pd.DataFrame([
    {
        "run": r["run"],
        "reward_corr": round(r["reward_mean_corr"], 4),
        "speed_corr": round(r["speed_mean_corr"], 4),
        "progress_corr": round(r["progress_delta_mean_corr"], 4),
        "steer_corr": round(r["steer_mean_corr"], 4),
        "offtrack_acc": round(r["offtrack_mean_accuracy"], 4),
    }
    for r in summary["world_model_reward_faithfulness"]
])

## 5. RSSM Architecture

Loads the model class from . No checkpoint, no gym.

In [ ]:
import torch
sys.path.insert(0, ".")
from world_model.models import RSSMSequence

model = RSSMSequence(action_dim=3, hidden_dim=512, stochastic_dim=32, embedding_dim=512)
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total:,}")
print(f"Trainable parameters: {trainable:,}")
print()
print(model)

## 6. World Model Hallucination Videos

**Video 3** — early RSSM hallucination (before telemetry pivot).
**Video 4** — P5 checkpoint: cleaner geometry, sharp-turn pose evolution still the open problem.

In [ ]:
for label, path in [
    ("Video 3 — Early RSSM hallucination", "docs/assets/video3.mp4"),
    ("Video 4 — P5 checkpoint hallucination", "docs/assets/video4.mp4"),
]:
    print(label)
    display(Video(path, embed=True, html_attributes="controls loop width=640"))
    print()